In [2]:
profession_list = ['Accountant',
 'Actor',
 'Actuary',
 'Administrative Assistant',
 'Administrator',
 'Air Traffic Controller',
 'Animal Trainer',
 'Anthropologist',
 'Appraiser',
 'Archaeologist',
 'Architect',
 'Archivist',
 'Art Director',
 'Artist',
 'Astronaut',
 'Astronomer',
 'Athlete',
 'Audio Technician',
 'Auditor',
 'Automotive Designer',
 'Baker',
 'Banker',
 'Bankruptcy Specialist',
 'Barber',
 'Barista',
 'Bartender',
 'Basketball player',
 'Biologist',
 'Biomedical Engineer',
 'Blacksmith',
 'Bodyguard',
 'Bounty Hunter',
 'Boxer',
 'Brand Manager',
 'Brewer',
 'Bricklayer',
 'Broker',
 'Builder',
 'Butcher',
 'CEO',
 'Carer',
 'Carpenter',
 'Cartographer',
 'Cashier',
 'Chef',
 'Chemical Engineer',
 'Chemist',
 'Chiropractor',
 'Civil Engineer',
 'Claims Adjuster',
 'Cleaner',
 'Clerk',
 'Coach',
 'Comedian',
 'Compliance Officer',
 'Composer',
 'Conservation Officer',
 'Construction Worker',
 'Copywriter',
 'Court Reporter',
 'Crime Scene Investigator',
 'Customer Support Specialist',
 'DJ',
 'Dancer',
 'Data Scientist',
 'Database Administrator',
 'Debt Counselor',
 'Dentist',
 'Detective',
 'Development Officer',
 'Dietitian',
 'Director',
 'Doctor',
 'Dog Walker',
 'Draughtsperson',
 'Driver',
 'Economist',
 'Editor',
 'Electrician',
 'Emergency Management Specialist',
 'Entrepreneur',
 'Environmental Engineer',
 'Ergonomist',
 'Estate Planner',
 'Event Coordinator',
 'Executive Assistant',
 'Exterminator',
 'Facilities Manager',
 'Farmer',
 'Fashion Designer',
 'Firefighter',
 'Fishmonger',
 'Flight Attendant',
 'Florist',
 'Football player',
 'Forklift Operator',
 'Gardener',
 'Geologist',
 'Graphic Designer',
 'Grocer',
 'Hair dresser',
 'Handyperson',
 'Health Inspector',
 'Historian',
 'Hotel Concierge',
 'Hotel Manager',
 'Human Resources Specialist',
 'IT Support Specialist',
 'Illustrator',
 'Industrial Designer',
 'Insurance Underwriter',
 'Janitor',
 'Jeweller',
 'Journalist',
 'Judge',
 'Lawyer',
 'Librarian',
 'Lifeguard',
 'Loan Officer',
 'Logger',
 'Logistics Manager',
 'Magician',
 'Makeup Artist',
 'Marine Biologist',
 'Marketing Manager',
 'Masseur',
 'Mathematician',
 'Mayor',
 'Mechanic',
 'Meteorologist',
 'Midwife',
 'Miner',
 'Model',
 'Musician',
 'News Reader',
 'Nurse',
 'Nutritionist',
 'Oceanographer',
 'Office Assistant',
 'Operations Manager',
 'Optician',
 'Painter',
 'Paralegal',
 'Paramedic',
 'Park Ranger',
 'Payroll Specialist',
 'Personal Trainer',
 'Pharmacist',
 'Photographer',
 'Physicist',
 'Pilot',
 'Plumber',
 'Police Officer',
 'Politician',
 'Postal Worker',
 'Priest',
 'Procurement Officer',
 'Professor',
 'Property Manager',
 'Psychologist',
 'Quality Assurance Inspector',
 'Real Estate Agent',
 'Receptionist',
 'Researcher',
 'Roofer',
 'Safety Inspector',
 'Sailor',
 'Salesperson',
 'Scientist',
 'Security Officer',
 'Shopkeeper',
 'Singer',
 'Skier',
 'Social Worker',
 'Software Engineer',
 'Soldier',
 'Sound Engineer',
 'Statistician',
 'Street Vendor',
 'Surfer',
 'Surgeon',
 'Swimmer',
 'Tailor',
 'Tattoo Artist',
 'Teacher',
 'Technician',
 'Tennis Player',
 'Therapist',
 'Translator',
 'Umpire',
 'Urban Planner',
 'Usher',
 'Veterinarian',
 'Videographer',
 'Waiter',
 'Waste Collection Worker',
 'Welder',
 'Wholesaler',
 'Writer',
 'Zoologist']

# prompt_templates = ["Male {object}", "Female {object}"]
prompt_templates = None # Gender invariant prompts are being created below via make_prompt to assign correct articles

def choose_article(noun: str) -> str:
    """
    Returns 'a' or 'an' based on the first letter of the noun.
    Sufficient for occupation names used in CLIP prompts.
    """
    return "an" if noun[0].lower() in "aeiou" else "a"

def make_prompt(profession: str) -> str:
    article = choose_article(profession)
    return f"A photo of {article} {profession.lower()}"


profession_list = [make_prompt(p) for p in profession_list]

print(profession_list[:45])

['A photo of an accountant', 'A photo of an actor', 'A photo of an actuary', 'A photo of an administrative assistant', 'A photo of an administrator', 'A photo of an air traffic controller', 'A photo of an animal trainer', 'A photo of an anthropologist', 'A photo of an appraiser', 'A photo of an archaeologist', 'A photo of an architect', 'A photo of an archivist', 'A photo of an art director', 'A photo of an artist', 'A photo of an astronaut', 'A photo of an astronomer', 'A photo of an athlete', 'A photo of an audio technician', 'A photo of an auditor', 'A photo of an automotive designer', 'A photo of a baker', 'A photo of a banker', 'A photo of a bankruptcy specialist', 'A photo of a barber', 'A photo of a barista', 'A photo of a bartender', 'A photo of a basketball player', 'A photo of a biologist', 'A photo of a biomedical engineer', 'A photo of a blacksmith', 'A photo of a bodyguard', 'A photo of a bounty hunter', 'A photo of a boxer', 'A photo of a brand manager', 'A photo of a bre

In [3]:
from pathlib import Path
import shutil
import json
import re
from concurrent.futures import ThreadPoolExecutor, as_completed

# -----------------------
# Configuration
# -----------------------

def iter_groups_from_jsonl(jsonl_path: Path, image_root: Path, max_per_prompt=None):
    """
    Yields:
        (group_name, lazy_items_function)
    where lazy_items_function returns [(Path, score, group_id), ...] when called
    """

    def resolve_src(r):
        raw = Path(r["image_path"])
        if raw.is_absolute():
            return raw
        return image_root / raw.parent.parent / f"{r['group_id']}_images" / raw.name

    with open(jsonl_path, "r", encoding="utf-8") as f:
        for line in f:
            try:
                obj = json.loads(line)
            except json.JSONDecodeError:
                continue

            prompt = re.sub(r'[<>:"/\\|?*]', '_', obj["prompt"]).strip().replace(" ", "_")
            results = obj.get("results", [])

            if max_per_prompt:
                results = results[:max_per_prompt]

            if not results:
                continue

            # Return a lambda that will resolve paths only when called
            def make_items_loader(results_copy):
                def load_items():
                    items = []
                    for r in results_copy:
                        p = resolve_src(r)
                        items.append((p, r["score"], r["group_id"]))
                    return items
                return load_items

            yield prompt, make_items_loader(results)

jsonl_1 = r"G:\Thesis\ImageRetrieval\Professions_125k_ISCO_Aligned\ISCO_aligned_125k_retrieval_results_batchsize_10.jsonl"
image_root = Path(r"E:\ImageRetrieval\Professions_125k_Cleaned")
output_root = Path(r"E:\ImageRetrieval\Professions_125k_ISCO_Aligned")
groups = iter_groups_from_jsonl(jsonl_1, image_root="")

MAX_COPY_WORKERS = 8     # Tune for your SSD (4–16 is typical)

# -----------------------
# Helpers
# -----------------------

def extract_occupation(prompt: str) -> str:
    prefixes = (
        "Male_",
        "Female_",
        "A_photo_of_an_",
        "A_photo_of_a_",
    )
    for p in prefixes:
        if prompt.startswith(p):
            return prompt[len(p):].upper()
    raise ValueError(f"Unrecognized prompt format: {prompt}")

def load_processed_keys(valid_txt_path: Path) -> set[str]:
    """
    Loads processed keys from valid.txt so reruns resume safely.
    Stored key format:
        {group_id}_{image_name}
    """
    processed = set()
    if not valid_txt_path.exists():
        return processed

    for line in valid_txt_path.read_text(encoding="utf-8").splitlines():
        parts = line.split("_", 2)
        if len(parts) == 3:
            _, group_id, image_name = parts
            processed.add(f"{group_id}_{image_name}")
    return processed

def load_invalid_keys(invalid_txt_path: Path) -> set[str]:
    """
    Loads invalid suffix keys from a source invalid.txt file.

    Expected filename format:
        <similarity>_<group_id>_<image_name>.jpg

    Returned key format:
        {group_id}_{image_name}
    """
    invalid = set()
    if not invalid_txt_path.exists():
        return invalid

    for line in invalid_txt_path.read_text(encoding="utf-8").splitlines():
        parts = line.split("_", 2)
        if len(parts) == 3:
            _, group_id, image_name = parts
            invalid.add(f"{group_id}_{image_name}")
    return invalid

def build_suffix_index(group_dir: Path) -> dict[str, Path]:
    """
    Builds:
        _{group_id}_{image_name} -> Path
    from filenames:
        <similarity>_<group_id>_<image_name>.jpg
    """
    index = {}

    for p in group_dir.iterdir():
        if not p.is_file():
            continue

        parts = p.name.split("_", 1)
        if len(parts) != 2:
            continue

        suffix = "_" + parts[1]
        index[suffix] = p

    return index

def copy_job(src: Path, dst: Path):
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)

# -----------------------
# Main processing loop
# -----------------------

for group_name, items_loader in groups:
    # ---- Remove
    fixed_professions = [p.replace(" ", "_") for p in profession_list[:48]]
    if group_name in fixed_professions:
        continue
    # ----- Remove till here
    print(f"\nProcessing group: {group_name}")

    occupation = extract_occupation(group_name)
    items = items_loader()

    # ---- Output directories ----
    group_out_dir = output_root / group_name
    facemesh_out_dir = group_out_dir / "facemesh"
    group_out_dir.mkdir(parents=True, exist_ok=True)
    facemesh_out_dir.mkdir(parents=True, exist_ok=True)

    valid_txt_path = group_out_dir / "valid.txt"
    invalid_txt_path = group_out_dir / "invalid.txt"

    processed_keys = load_processed_keys(valid_txt_path)
    print(f"  Loaded {len(processed_keys)} previously processed images.")

    copy_futures = []
    newly_valid = []
    newly_invalid = []

    # Track all suffixes found in Male/Female
    found_keys = set()

    # Aggregate invalid keys from source directories
    source_invalid_keys = set()

    with ThreadPoolExecutor(max_workers=MAX_COPY_WORKERS) as pool:

        # Male has priority over Female
        for new_group_name in [f"Male_{occupation}", f"Female_{occupation}"]:
            group_dir = image_root / new_group_name
            if not group_dir.exists():
                print(f"  Skipping — directory not found: {group_dir}")
                continue

            print(f"  Indexing: {group_dir}")
            suffix_index = build_suffix_index(group_dir)

            # Load invalid.txt from this source directory
            src_invalid_txt = group_dir / "invalid.txt"
            src_invalid_keys = load_invalid_keys(src_invalid_txt)
            if src_invalid_keys:
                print(f"  Loaded {len(src_invalid_keys)} invalid keys from {src_invalid_txt}")
            source_invalid_keys |= src_invalid_keys

            for src_path, similarity, group_id in items:
                image_name = src_path.name
                key = f"{group_id}_{image_name}"

                # Skip if already copied from previous gender or previous run
                if key in processed_keys:
                    found_keys.add(key)
                    continue

                suffix = f"_{group_id}_{image_name}"
                match = suffix_index.get(suffix)
                if match is None:
                    continue

                found_keys.add(key)

                # ---- Facemesh source ----
                face_match = (
                    match.parent / "facemesh" /
                    (match.stem + "_face.png")
                )

                # ---- Build aligned filenames ----
                # new_name = f"{round(similarity, 3)}_{group_id}_{image_name}"
                new_name = f"{similarity:.3f}_{group_id}_{image_name}"
                final_img_name = Path(new_name).stem + ".jpg"
                final_face_name = Path(new_name).stem + "_face.png"

                target_path = group_out_dir / final_img_name
                face_target_path = facemesh_out_dir / final_face_name

                # ---- Schedule async copies ----
                copy_futures.append(pool.submit(copy_job, match, target_path))

                if face_match.exists():
                    copy_futures.append(pool.submit(copy_job, face_match, face_target_path))

                newly_valid.append(final_img_name)
                processed_keys.add(key)

        # ---- Wait for all copies to finish ----
        for f in as_completed(copy_futures):
            f.result()   # propagate exceptions

    # -----------------------
    # Handle INVALID propagation
    # -----------------------

    # Any JSONL item not found in Male/Female
    for src_path, similarity, group_id in items:
        image_name = src_path.name
        key = f"{group_id}_{image_name}"

        if key in found_keys:
            continue

        # If marked invalid in either source directory
        if key in source_invalid_keys:
            # new_name = f"{round(similarity, 3)}_{group_id}_{image_name}"
            new_name = f"{similarity:.3f}_{group_id}_{image_name}"
            final_invalid_name = Path(new_name).stem + ".jpg"
            newly_invalid.append(final_invalid_name)

    # -----------------------
    # Write outputs once
    # -----------------------

    if newly_valid:
        with open(valid_txt_path, "a", encoding="utf-8") as valid_f:
            for name in newly_valid:
                valid_f.write(name + "\n")

    if newly_invalid:
        # Deduplicate against existing invalid.txt
        existing_invalid = set()
        if invalid_txt_path.exists():
            existing_invalid = set(
                invalid_txt_path.read_text(encoding="utf-8").splitlines()
            )

        with open(invalid_txt_path, "a", encoding="utf-8") as invalid_f:
            for name in newly_invalid:
                if name not in existing_invalid:
                    invalid_f.write(name + "\n")

    print(f"  Copied {len(newly_valid)} valid images.")
    print(f"  Marked {len(newly_invalid)} invalid images.")

    # # Remove this break once validated
    # break


Processing group: A_photo_of_a_civil_engineer
  Loaded 0 previously processed images.
  Indexing: E:\ImageRetrieval\Professions_125k_Cleaned\Male_CIVIL_ENGINEER
  Loaded 103090 invalid keys from E:\ImageRetrieval\Professions_125k_Cleaned\Male_CIVIL_ENGINEER\invalid.txt
  Indexing: E:\ImageRetrieval\Professions_125k_Cleaned\Female_CIVIL_ENGINEER
  Loaded 97288 invalid keys from E:\ImageRetrieval\Professions_125k_Cleaned\Female_CIVIL_ENGINEER\invalid.txt
  Copied 12615 valid images.
  Marked 32754 invalid images.

Processing group: A_photo_of_a_claims_adjuster
  Loaded 0 previously processed images.
  Indexing: E:\ImageRetrieval\Professions_125k_Cleaned\Male_CLAIMS_ADJUSTER
  Loaded 113638 invalid keys from E:\ImageRetrieval\Professions_125k_Cleaned\Male_CLAIMS_ADJUSTER\invalid.txt
  Indexing: E:\ImageRetrieval\Professions_125k_Cleaned\Female_CLAIMS_ADJUSTER
  Loaded 108821 invalid keys from E:\ImageRetrieval\Professions_125k_Cleaned\Female_CLAIMS_ADJUSTER\invalid.txt
  Copied 10837 val

### Code to clean up the no longer needed directory and remove non facemesh images since these are simply duplicates of images from the Re-LAion-5B dataset.

In [4]:
import time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import Iterable
from tqdm import tqdm

# ---------------- Configuration ----------------

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tiff", ".tif"}
SKIP_DIR_TOKENS = {"facemesh"}

# Tune for your disk
MAX_WORKERS = 8

# ------------------------------------------------


def should_skip_path(path: Path) -> bool:
    """
    Returns True if the file is inside a directory that should be skipped.
    """
    for part in path.parts:
        part_l = part.lower()
        for token in SKIP_DIR_TOKENS:
            if token in part_l:
                return True
    return False


def iter_image_files(root: Path) -> Iterable[Path]:
    """
    Yield image files under root except those inside skipped directories.
    """
    for path in root.rglob("*"):
        if not path.is_file():
            continue
        if path.suffix.lower() not in IMAGE_EXTS:
            continue
        if should_skip_path(path):
            continue
        yield path


def delete_file(path: Path) -> bool:
    try:
        path.unlink()
        return True
    except Exception:
        return False


def delete_directory_images(dir_path: Path):
    """
    Deletes images inside one subdirectory and reports timing.
    """
    start = time.perf_counter()
    print(f"\n▶ Starting: {dir_path}")

    image_files = list(iter_image_files(dir_path))
    total = len(image_files)

    if total == 0:
        print(f"✓ Nothing to delete in {dir_path}")
        return 0, 0.0

    deleted = 0

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = [executor.submit(delete_file, p) for p in image_files]

        for f in tqdm(as_completed(futures), total=total, desc=f"Deleting {dir_path.name}", leave=False):
            if f.result():
                deleted += 1

    elapsed_min = (time.perf_counter() - start) / 60.0

    print(
        f"✓ Finished: {dir_path} | "
        f"Deleted: {deleted:,} files | "
        f"Time: {elapsed_min:.2f} min"
    )

    return deleted, elapsed_min


def clean_root(main_root: str):
    root = Path(main_root)

    if not root.exists():
        raise FileNotFoundError(root)

    print(f"\nScanning root: {root}\n")

    total_deleted = 0
    total_time = 0.0

    # Only process immediate subdirectories
    subdirs = sorted(p for p in root.iterdir() if p.is_dir())

    for idx, subdir in enumerate(subdirs, start=1):
        print(f"\n========== [{idx}/{len(subdirs)}] ==========")
        deleted, minutes = delete_directory_images(subdir)
        total_deleted += deleted
        total_time += minutes

    print("\n============================================")
    print(f"ALL DONE")
    print(f"Total deleted: {total_deleted:,}")
    print(f"Total time   : {total_time:.2f} minutes")
    print("============================================")


# ---------------- Entry Point ----------------

if __name__ == "__main__":
    import argparse

    # parser = argparse.ArgumentParser(
    #     description="Delete images per subdirectory with timing and progress logging."
    # )
    # parser.add_argument(
    #     "--root",
    #     required=True,
    #     help="Main directory (contains sub_dir_1, sub_dir_2, ...)"
    # )

    # args = parser.parse_args()
    # clean_root(args.root)

    root = r"E:\ImageRetrieval\Professions_125k_Cleaned"
    clean_root(root)



Scanning root: E:\ImageRetrieval\Professions_125k_Cleaned


========== [1/290] ==========

▶ Starting: E:\ImageRetrieval\Professions_125k_Cleaned\Female_Accountant


✓ Finished: E:\ImageRetrieval\Professions_125k_Cleaned\Female_Accountant | Deleted: 22,102 files | Time: 0.20 min

========== [2/290] ==========

▶ Starting: E:\ImageRetrieval\Professions_125k_Cleaned\Female_Actor


✓ Finished: E:\ImageRetrieval\Professions_125k_Cleaned\Female_Actor | Deleted: 58,635 files | Time: 0.57 min

========== [3/290] ==========

▶ Starting: E:\ImageRetrieval\Professions_125k_Cleaned\Female_Actuarial_Analyst


✓ Finished: E:\ImageRetrieval\Professions_125k_Cleaned\Female_Actuarial_Analyst | Deleted: 18,783 files | Time: 0.16 min

========== [4/290] ==========

▶ Starting: E:\ImageRetrieval\Professions_125k_Cleaned\Female_Actuary


✓ Finished: E:\ImageRetrieval\Professions_125k_Cleaned\Female_Actuary | Deleted: 32,498 files | Time: 0.35 min

========== [5/290] ==========

▶ Starting: E:\ImageRetrieval\Professions_125k_Cleaned\Female_Administrative_Assistant


...

✓ Finished: E:\ImageRetrieval\Professions_125k_Cleaned\Male_Meteorologist | Deleted: 35,057 files | Time: 0.92 min

========== [287/290] ==========

▶ Starting: E:\ImageRetrieval\Professions_125k_Cleaned\Male_Midwife


✓ Finished: E:\ImageRetrieval\Professions_125k_Cleaned\Male_Midwife | Deleted: 17,422 files | Time: 0.60 min

========== [288/290] ==========

▶ Starting: E:\ImageRetrieval\Professions_125k_Cleaned\Male_Miner


✓ Finished: E:\ImageRetrieval\Professions_125k_Cleaned\Male_Miner | Deleted: 19,325 files | Time: 0.65 min

========== [289/290] ==========

▶ Starting: E:\ImageRetrieval\Professions_125k_Cleaned\Male_Model


✓ Finished: E:\ImageRetrieval\Professions_125k_Cleaned\Male_Model | Deleted: 66,853 files | Time: 2.55 min

========== [290/290] ==========

▶ Starting: E:\ImageRetrieval\Professions_125k_Cleaned\Male_Musician


✓ Finished: E:\ImageRetrieval\Professions_125k_Cleaned\Male_Musician | Deleted: 37,062 files | Time: 1.28 min

ALL DONE
Total deleted: 9,060,632
Total time   : 324.33 minutes
